## Imports

In [99]:
import time
import matplotlib.pyplot as plt
import random
import gmpy2
import itertools
from tqdm.notebook import tqdm
import numpy as np
import sympy
import math
import cProfile

## Libs

In [84]:
def benchmark(factorization, lower, upper):
    p1 = sympy.randprime(10**lower, 10**upper)
    p2 = sympy.randprime(10**lower, 10**upper)

    target_semi_prime = p1 * p2

    print(f"Challenge: Factor the following semi-prime:")
    print(f"{target_semi_prime}")
    print(f"Magnitude: 10^{len(str(target_semi_prime))}")

    start = time.perf_counter()
    factor = factorization(target_semi_prime)
    end = time.perf_counter()
    print(f"Time taken: {end - start:.4f} seconds")
    print(f"Check N % f == {target_semi_prime % factor}")
    print(f"Factor: {factor}")

## Naive factoring

In [85]:
def naiveFactoring(N):
    for i in range(2, int(N**0.5) + 1):
        if N % i == 0:
            return i
    return None

benchmark(naiveFactoring, 3, 4)

Challenge: Factor the following semi-prime:
36839941
Magnitude: 10^8
Time taken: 0.0002 seconds
Check N % f == 0
Factor: 5303


## Pollar Rho

In [95]:
def pollardRho(N):
    if N % 2 == 0:
        return 2
    for c in range(1, 10):
        def f(x, c, n): return (pow(x, 2, n) + c) % n

        x = 2
        T = x
        H = x
        d = 1

        while d == 1:
            T = f(T, c, N)
            H = f(f(H, c, N), c, N)
            d = math.gcd(abs(T - H), N)

            if d == N:
                break
            if d > 1:
                return d
    return None

benchmark(pollardRho, 12, 13)

Challenge: Factor the following semi-prime:
36021091719598852955333263
Magnitude: 10^26
Time taken: 6.1078 seconds
Check N % f == 0
Factor: 7129808923483


## Pollar Rho Opt.

In [ ]:
def pollardRhoBrent(N):
    if N % 2 == 0:
        return 2
    y, c, m = 2, 1, 128
    g, r, q = 1, 1, 1
    x, ys = 0, 0

    while g == 1:
        x = y
        for i in range(r):
            y = (y * y + c) % N

        k = 0
        while k < r and g == 1:
            ys = y
            limit = min(m, r - k)
            for i in range(limit):
                y = (y * y + c) % N
                diff = x - y
                if diff < 0:
                    diff = -diff
                q = (q * diff) % N
            g = math.gcd(q, N)
            k += m
        r <<= 1

    if g == N:
        while True:
            ys = (ys * ys + c) % N
            g = math.gcd(abs(x - ys), N)
            if g > 1:
                break
    return g

benchmark(pollardRho, 13, 14)

Challenge: Factor the following semi-prime:
2695280762882812138463547869
Magnitude: 10^28
Time taken: 11.3027 seconds
Check N % f == 0
Factor: 33253703478341
         25364652 function calls (25364599 primitive calls) in 11.303 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    1.591    1.591    6.656    6.656 2375465233.py:1(pollardRho)
  9511098    1.918    0.000    5.212    0.000 2375465233.py:5(f)
        1    0.000    0.000    6.657    6.657 897423386.py:1(benchmark)
        4    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        7    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.000    0.000    6.657    6.657 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 <string>:2(__init__)
        4    0.000    0.000    0.000    0.000 __init__.py:183(dumps)
        1    0.000    0.000    0.005    0.005 _base.py:337(_invoke